In [1]:
import pandas as pd
import numpy as np
import datetime
from io import BytesIO
from zipfile import ZipFile
from urllib.request import urlopen

In [2]:
# https://covid19.healthdata.org/thailand?view=infections-testing&tab=trend&test=infections

# est_infections_mean = Estimated infections are the number of people we estimate are infected with COVID-19 each day, including those not tested

In [5]:
covid_zip = "https://ihmecovid19storage.blob.core.windows.net/latest/ihme-covid19.zip"
resp = urlopen(covid_zip)
zipfile = ZipFile(BytesIO(resp.read()))
file_name = [x for x in zipfile.namelist() if "reference_hospitalization_all_locs" in x]
covid = pd.read_csv(zipfile.open(file_name[0]))
covid = covid[["date", "location_name", "confirmed_infections", "est_infections_mean"]]
covid = covid[
    covid["location_name"].isin(["Thailand", "Malaysia", "Singapore", "Indonesia"])
]
covid["date"] = pd.to_datetime(covid["date"])

In [6]:
covid

,date,location_name,confirmed_infections,est_infections_mean
101534,2020-02-10,Indonesia,NaN,0.000000
101535,2020-02-11,Indonesia,NaN,5.335421
101536,2020-02-12,Indonesia,NaN,10.316461
101537,2020-02-13,Indonesia,NaN,19.149550
101538,2020-02-14,Indonesia,NaN,34.176191
...,...,...,...,...
226781,2021-11-27,Thailand,NaN,106978.251693
226782,2021-11-28,Thailand,NaN,108252.194159
226783,2021-11-29,Thailand,NaN,109145.460011
226784,2021-11-30,Thailand,NaN,109031.536602


In [7]:
covid["id_shop_name"] = np.select(
    [
        covid["location_name"] == "Thailand",
        covid["location_name"] == "Singapore",
        covid["location_name"] == "Malaysia",
        covid["location_name"] == "Indonesia",
    ],
    ["TH", "SG", "MY", "ID"],
)

In [8]:
df_group = (
    covid.groupby(["date", "id_shop_name"])["est_infections_mean"].sum().reset_index()
)

In [9]:
df_group

,date,id_shop_name,est_infections_mean
0,2020-02-04,MY,21.425826
1,2020-02-04,SG,16.705412
2,2020-02-04,TH,26.637842
3,2020-02-05,MY,21.956281
4,2020-02-05,SG,16.443418
...,...,...,...
2657,2021-11-30,TH,109031.536602
2658,2021-12-01,ID,65359.811634
2659,2021-12-01,MY,654.616515
2660,2021-12-01,SG,92.405014


In [10]:
df_group.rename(columns={"date": "start_date_weekly"}, inplace=True)

In [11]:
df_group["start_date_weekly"] = pd.to_datetime(df_group["start_date_weekly"])

In [12]:
df_group[df_group["start_date_weekly"] == "2021-08-08"]

,start_date_weekly,id_shop_name,est_infections_mean
2198,2021-08-08,ID,273190.496674
2199,2021-08-08,MY,115803.547095
2200,2021-08-08,SG,106.494619
2201,2021-08-08,TH,69100.068987


In [13]:
df_group.to_csv(
    "s3://hal-bi-bucket/data_science/dfm/offline_clothing_v2/data/covid_projections_offline.csv",
    index=False,
)